In [5]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from src.data_manipulation import add_era_to_all, add_round_group, round_group_summary

In [6]:
# import data
import sys
sys.path.append('..')

raw_data = pd.read_csv('../data/uefa_results.csv')
country_stats = pd.read_pickle('../data/processed/country_stats_complete.pkl')
country_season_highest = pd.read_pickle('../data/processed/country_season_highest.pkl')
country_season_competition_highest = pd.read_pickle('../data/processed/country_season_competition_highest.pkl')

In [7]:
# adding the Golden Era label to each dataframe

rd, cs, csh, csch = add_era_to_all(
    raw_data, 
    country_stats,
    country_season_highest, 
    country_season_competition_highest,
    golden_start="1987/88",
    golden_end="1999/00",
)


  Era definition : 1987/88  →  1999/00
  Era                     Seasons      First       Last
  ----------------------------------------------------
  Pre-Golden Era               18    1969/70    1986/87
  Golden Era                   13    1987/88    1999/00
  Post-Golden Era              26    2000/01    2025/26



In [8]:
# Round grouping on raw_data only
rd = add_round_group(rd)
round_group_summary(rd) # sanity check


  [round_group] All round labels mapped successfully (13872 / 13872 rows).

  Round group             Count       %
  --------------------------------------
  stage                  12,747   91.9%
  Round of 16               119    0.9%
  Quarter Finals            572    4.1%
  Semi Finals               288    2.1%
  Final                     146    1.1%
  TOTAL                  13,872  100.0%



# Chapter 1: Comparing Football Italian Clubs in Europe dufint different eras

## Formal Framework: What we are testing

The initial question if the Italian Football Club went through a "golden-age" can be broken down into 2 independent, testifiable hypotheses:

- $H_{1}$ (level) - Italy's match-level performance (win rate, points, goal difference) was higher during the Golden Era than in the other two eras.
- $H_{2}$ (depth) -  Italy reached deeper knockout rounds during the Golden Era than in the other two eras.

Proposed metrics:

| Pillar | Metric | Source column | Why |
|:-------|:-------|:--------------|:----|
|Match performance  | Win rate | `win_rate` | Direct efficiency measure |
|Match performance  | points per game (*PPG*) | `ppg_3`/`num_teams` | Normalise per quota size |
|Match performance  | Goal Difference per Game (*GD per game*) | `gdpg` | Quality of wins, not just binary |
|Tournament depth  | Avg deepest round | `highest_round` | How far teams progressed |
|Tournament depth  | Finals/semis count | `highest_round` | Absolute peak moments |


## Statistical Approach

Each metric produces one value per season. With less than 30 seasons per era at most, samples are small and unlikely to be normal, so we use non-parametric tests throughout.

- [Mann-Whitney $U$ test](https://en.wikipedia.org/wiki/Mann%E2%80%93Whitney_U_test) - pairwise: Golden vs Pre, Golden vs Post. One-tailed (alternative="greater") since we are testing a directional hypothesis.
- [Cohen's $d$](https://en.wikipedia.org/wiki/Effect_size#Cohen's_d) - effect size alongside every *p-value*. A statistically significant result with a tiny effect size is not analytically meaningful: if the Mann-Whitney U test will inform if there is a statistically significant difference, the effect size $d$ will give us an indication about how "strong" the effect of the difference will be.
- Significance threshold — $\alpha$ = 0.05, with 0.1 as a weak signal worth noting. 

### a. Filter to Italy

The first step involves building 2 data structures producing two DataFrames relative to Italian clubs only:
  - `italy_cs` : from `country_stats` (see data_wrangling.ipynb notebook for reference)
  - `italy_csh` : from `country_season_highest` (see data_wrangling.ipynb notebook for reference)
 
Case-insensitive match on the country column so 'Ita', 'ITA', 'ita' all work regardless of how the data was loaded.

In [9]:
import pandas as pd
 
def filter_country(
        cs:  pd.DataFrame,
        csh: pd.DataFrame,
        country_col: str = "country",
        country_name: str = "Ita",
        ) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Filter country_stats and country_season_highest to a single country.
 
    Parameters
    ----------
    cs           : country_stats DataFrame  (must already have 'era' column)
    csh          : country_season_highest DataFrame  (must already have 'era')
    country_col  : name of the country column  (default 'country')
    country_name : country to filter to         (default 'Ita')
 
    Returns
    -------
    italy_cs, italy_csh  — filtered copies, index reset
    """
    name_upper = country_name.strip().upper()
 
    italy_cs  = (cs [cs [country_col].str.upper() == name_upper]
                 .copy().reset_index(drop=True))
    italy_csh = (csh[csh[country_col].str.upper() == name_upper]
                 .copy().reset_index(drop=True))
 
    # ── Basic coverage check ──────────────────────────────────────────────────
    print(f"\n  Filtering to: {country_name}")
    print(f"  country_stats          rows : {len(italy_cs)}")
    print(f"  country_season_highest rows : {len(italy_csh)}")
 
    for label, df in [("country_stats", italy_cs),
                      ("country_season_highest", italy_csh)]:
        if "era" not in df.columns:
            print(f"  WARNING: 'era' column missing from {label} — "
                  "run add_era_column() first (step0_era.py)")
        else:
            era_counts = df["era"].value_counts().sort_index()
            print(f"  {label} era breakdown:")
            for era, n in era_counts.items():
                print(f"      {era:<22}: {n} seasons")
 
    return italy_cs, italy_csh

In [10]:
italy_cs, italy_csh = filter_country(cs, csh, country_name='Ita')


  Filtering to: Ita
  country_stats          rows : 57
  country_season_highest rows : 57
  country_stats era breakdown:
      Pre-Golden Era        : 18 seasons
      Golden Era            : 13 seasons
      Post-Golden Era       : 26 seasons
  country_season_highest era breakdown:
      Pre-Golden Era        : 18 seasons
      Golden Era            : 13 seasons
      Post-Golden Era       : 26 seasons


### b. Derived metrics: adding the normalised metrics

Add normalised `per-team` metrics to `italy_cs`. We take the points per game (3 points and 2 points), the goal difference per game and the percentage of won matches and normalise them for the number of teams for that season

> Some of this metrics may be confusing, clarify when they have been used, how and why.

In [11]:
def add_derived_metrics(italy_cs: pd.DataFrame) -> pd.DataFrame:
    """
    Add normalised per-team metrics to italy_cs.
 
    Parameters
    ----------
    italy_cs : Italy-filtered country_stats (output of step1a filter_italy)
 
    Returns
    -------
    Copy of italy_cs with three new columns:
        ppg3_pt, ppg2_pt, gdpg_pt
    """
    _REQUIRED = ["num_teams", "ppg_3", "ppg_2", "gdpg", "win_rate"]
    
    missing = [c for c in _REQUIRED if c not in italy_cs.columns]
    if missing:
        raise KeyError(f"Missing columns in italy_cs: {missing}")
 
    out = italy_cs.copy()
 
    # Guard: avoid division by zero if num_teams is 0 or NaN
    teams = out["num_teams"].replace(0, np.nan)
 
    out["ppg3_pt"] = out["ppg_3"] / teams
    out["ppg2_pt"] = out["ppg_2"] / teams
    out["gdpg_pt"] = out["gdpg"]  / teams
 
    # ── Sanity check: print a small preview ───────────────────────────────────
    preview_cols = ["season", "era", "num_teams",
                    "win_rate", "ppg3_pt", "gdpg_pt"]
    print("\n  Derived metrics preview (first 5 rows):")
    print(out[preview_cols].head().to_string(index=False))
 
    null_counts = out[["ppg3_pt", "ppg2_pt", "gdpg_pt"]].isnull().sum()
    if null_counts.any():
        print(f"\n  WARNING - NaNs introduced (likely num_teams == 0):")
        print(null_counts[null_counts > 0].to_string())
 
    return out

In [12]:
italy_cs = add_derived_metrics(italy_cs)


  Derived metrics preview (first 5 rows):
 season            era  num_teams  win_rate  ppg3_pt  gdpg_pt
1969/70 Pre-Golden Era          7  0.523810 0.258503 0.064626
1970/71 Pre-Golden Era          6  0.423077 0.269231 0.102564
1971/72 Pre-Golden Era          6  0.461538 0.282051 0.145299
1972/73 Pre-Golden Era          6  0.593750 0.328125 0.119792
1973/74 Pre-Golden Era          6  0.428571 0.238095 0.015873


### c. Per-era summary table

We now compute mean, standard deviation, median, min and max for each metric across the three eras. One row per era, one column group per metric.

In [13]:
import pandas as pd
import numpy as np

def era_summary_table(italy_cs: pd.DataFrame) -> pd.DataFrame:
    """
    Return a tidy summary DataFrame: one row per era, stats for each metric.

    Parameters
    ----------
    italy_cs : Italy country_stats with 'era', 'win_rate', 'ppg3_pt', 'gdpg_pt'
               (output of step1b add_derived_metrics)

    Returns
    -------
    summary : DataFrame with MultiIndex columns  (metric, stat)
              Rows are ordered  Pre → Golden → Post
    """

    # ── Era labels (single source of truth) ──────────────────────────────────────
    ERA_PRE    = "Pre-Golden Era"
    ERA_GOLDEN = "Golden Era"
    ERA_POST   = "Post-Golden Era"

    # Ordered list – useful for sorting / CategoricalDtype later
    ERA_ORDER = [ERA_PRE, ERA_GOLDEN, ERA_POST]

    # Metrics to summarise and their display labels
    METRICS = {
        "win_rate" : "Win rate",
        "ppg3_pt"  : "PPG (3pt) per team",
        "gdpg_pt"  : "GDpG per team",
    }

    missing = [c for c in list(METRICS) + ["era"]
               if c not in italy_cs.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}. "
                       "Run step1a and step1b first.")

    records = []
    for era in ERA_ORDER:
        sub = italy_cs[italy_cs["era"] == era]
        row = {"era": era, "n_seasons": len(sub)}
        for col in METRICS:
            vals = sub[col].dropna()
            row[f"{col}_mean"]   = vals.mean()
            row[f"{col}_std"]    = vals.std(ddof=1)
            row[f"{col}_median"] = vals.median()
            row[f"{col}_min"]    = vals.min()
            row[f"{col}_max"]    = vals.max()
        records.append(row)

    summary = pd.DataFrame(records).set_index("era")

    # ── Pretty-print ──────────────────────────────────────────────────────────
    print("\n  Per-era summary  (Italy · European competitions)")
    print(f"  {'Era':<22}  {'N':>2}  "
          + "  ".join(f"{'  ' + lbl + '  ':^28}" for lbl in METRICS.values()))
    print(f"  {'':22}  {'':2}  "
          + "  ".join(f"{'mean':>7} {'± std':>7} {'median':>8}"
                      for _ in METRICS))
    print("  " + "-" * 90)

    for era in ERA_ORDER:
        if era not in summary.index:
            continue
        r   = summary.loc[era]
        n   = int(r["n_seasons"])
        row = f"  {era:<22}  {n:>2}  "
        for col in METRICS:
            m  = r[f"{col}_mean"]
            s  = r[f"{col}_std"]
            md = r[f"{col}_median"]
            row += f"  {m:>7.3f} {s:>7.3f} {md:>8.3f}  "
        print(row)

    print()
    return summary


def era_delta_table(summary: pd.DataFrame) -> pd.DataFrame:
    """
    Show how much Golden Era differs from Pre and Post (absolute and %).

    Parameters
    ----------
    summary : output of era_summary_table()

    Returns
    -------
    delta : DataFrame with rows  (vs Pre-Golden Era, vs Post-Golden Era)
            and columns for each metric's absolute and % difference
    """
    # Metrics to summarise and their display labels
    METRICS = {
        "win_rate" : "Win rate",
        "ppg3_pt"  : "PPG (3pt) per team",
        "gdpg_pt"  : "GDpG per team",
    }

    if "Golden Era" not in summary.index:
        print("  No Golden Era rows found — skipping delta table.")
        return pd.DataFrame()

    golden = summary.loc["Golden Era"]
    rows   = []

    for compare_era in ["Pre-Golden Era", "Post-Golden Era"]:
        if compare_era not in summary.index:
            continue
        other = summary.loc[compare_era]
        row   = {"comparison": f"Golden vs {compare_era}"}
        for col in METRICS:
            g = golden[f"{col}_mean"]
            o = other [f"{col}_mean"]
            row[f"{col}_diff"] = g - o
            row[f"{col}_pct"]  = 100 * (g - o) / o if o != 0 else np.nan
        rows.append(row)

    delta = pd.DataFrame(rows).set_index("comparison")

    print("  Golden Era delta vs other eras:")
    print(f"  {'Comparison':<34}  "
          + "  ".join(f"{lbl:^22}" for lbl in METRICS.values()))
    print(f"  {'':34}  "
          + "  ".join(f"{'diff':>9} {'%':>9}" for _ in METRICS))
    print("  " + "-" * 80)

    for idx, r in delta.iterrows():
        row = f"  {idx:<34}  "
        for col in METRICS:
            d   = r[f"{col}_diff"]
            pct = r[f"{col}_pct"]
            row += f"  {d:>+9.3f} {pct:>+8.1f}%  "
        print(row)

    print()
    return delta

In [14]:
summary = era_summary_table(italy_cs)


  Per-era summary  (Italy · European competitions)
  Era                      N            Win rate                 PPG (3pt) per team              GDpG per team        
                                 mean   ± std   median     mean   ± std   median     mean   ± std   median
  ------------------------------------------------------------------------------------------
  Pre-Golden Era          18      0.461   0.084    0.435      0.303   0.081    0.282      0.088   0.053    0.079  
  Golden Era              13      0.545   0.070    0.553      0.275   0.038    0.278      0.114   0.040    0.107  
  Post-Golden Era         26      0.460   0.041    0.462      0.238   0.030    0.236      0.061   0.026    0.060  



The above table summarise the mean, median and standard deviation for the 3 metrics measuring match performances (namely Win Rate, PPG (3pm) per team, GDpG per team) divided per era. If we look at the Win Rate, we can see that pre and post golden era, the mean is preatty identical, with just the standard deviation that is different (but we also have to address the fact that the number of season considered in the Pre-golden era is less than the Post-Golden era and we know that standard deviation depends on the square root of the number of samples). The Win rate in the Golden Era is higher with respect to both Pre and Post considered era.

In [15]:
delta = era_delta_table(summary)

  Golden Era delta vs other eras:
  Comparison                                 Win rate           PPG (3pt) per team        GDpG per team     
                                           diff         %       diff         %       diff         %
  --------------------------------------------------------------------------------
  Golden vs Pre-Golden Era                 +0.084    +18.3%       -0.028     -9.3%       +0.027    +30.2%  
  Golden vs Post-Golden Era                +0.085    +18.4%       +0.037    +15.5%       +0.053    +87.9%  



The table shows the difference in mean. We can see that, approximately, we have 18% more win rate in the Golden era with respect to Pre and Post era. The goal difference is even more: +30% compared with Pre-golden and +87%. It's worth noticing that when we normalize per team we have to be careful and considering the different number of teams in the different eras. 

### d. Tournament depth

The other pillar used to evaluate the overall performance is the *Tournament depth*. Analysing the tournment depth means considering the average, meadian and max deepest rond reached and counting how many seasons each era produced a Final, a Semi-final and a quarter final stage 

- The 'highest_round' column contains grouped labels: 'stage', 'Round of 16', 'Quarter Finals', 'Semi Finals', 'Final'
- We add an ordered Categorical here (just for this DataFrame and this step) so that max(), sort, and comparisons work correctly.

In [16]:
import pandas as pd
# from step0_round import ROUND_GROUPS   # ordered list, stage → Final
# from step0_era   import ERA_ORDER

# ── Era labels (single source of truth) ──────────────────────────────────────
ERA_PRE    = "Pre-Golden Era"
ERA_GOLDEN = "Golden Era"
ERA_POST   = "Post-Golden Era"

# Ordered list – useful for sorting / CategoricalDtype later
ERA_ORDER = [ERA_PRE, ERA_GOLDEN, ERA_POST]

# ── Ordered group labels (single source of truth) ─────────────────────────────
ROUND_GROUPS = ["stage", "Round of 16", "Quarter Finals", "Semi Finals", "Final"]


def _apply_round_order(italy_csh: pd.DataFrame,
                        round_col: str = "highest_round") -> pd.DataFrame:
    """
    Cast highest_round to an ordered Categorical on a copy of the DataFrame.
    Returns the enriched copy.
    """
    out = italy_csh.copy()
    out[round_col] = pd.Categorical(
        out[round_col],
        categories=ROUND_GROUPS,
        ordered=True,
    )
    # Add numeric code for mean / std calculations (stage=0 … Final=4)
    out["round_ord"] = out[round_col].cat.codes.replace(-1, pd.NA)
    return out


def depth_summary_table(italy_csh: pd.DataFrame,
                        round_col: str = "highest_round") -> pd.DataFrame:
    """
    Per-era average, median and maximum deepest round reached by Italy.

    Parameters
    ----------
    italy_csh : Italy-filtered country_season_highest with 'era' column
    round_col : column holding the grouped round label

    Returns
    -------
    summary DataFrame indexed by era
    """
    df = _apply_round_order(italy_csh, round_col)

    records = []
    for era in ERA_ORDER:
        sub  = df[df["era"] == era]
        ords = sub["round_ord"].dropna()
        records.append({
            "era"        : era,
            "n_seasons"  : len(sub),
            "avg_ord"    : ords.mean(),
            "median_ord" : ords.median(),
            "max_ord"    : ords.max(),
            # Human-readable labels for the most common and best round
            "avg_round"  : (ROUND_GROUPS[round(ords.mean())]
                            if not ords.empty and not pd.isna(ords.mean())
                            else "—"),
            "best_round" : (ROUND_GROUPS[int(ords.max())]
                            if not ords.empty and not pd.isna(ords.max())
                            else "—"),
        })

    summary = pd.DataFrame(records).set_index("era")

    print("\n  Tournament depth by era  (Italy)")
    print(f"  {'Era':<22}  {'N':>2}  {'Avg ord':>8}  "
          f"{'Avg round':<16}  {'Best round':<16}")
    print("  " + "-" * 72)
    for era in ERA_ORDER:
        if era not in summary.index:
            continue
        r = summary.loc[era]
        print(f"  {era:<22}  {int(r['n_seasons']):>2}  "
              f"{r['avg_ord']:>8.2f}  "
              f"{r['avg_round']:<16}  {r['best_round']:<16}")
    print()
    return summary


def milestone_counts(italy_csh: pd.DataFrame,
                     round_col: str = "highest_round") -> pd.DataFrame:
    """
    Count how many seasons Italy reached each milestone per era.

    Milestones: Final, Semi Finals, Quarter Finals, Round of 16
    (each season contributes to the deepest milestone reached only)

    Returns
    -------
    pivot DataFrame: rows = era, columns = milestone rounds, values = count
    """
    df  = _apply_round_order(italy_csh, round_col)
    milestones = ["Final", "Semi Finals", "Quarter Finals", "Round of 16"]

    records = []
    for era in ERA_ORDER:
        sub = df[df["era"] == era]
        row = {"era": era, "n_seasons": len(sub)}
        for m in milestones:
            row[m] = (sub[round_col] == m).sum()
        row["stage_only"] = (sub[round_col] == "stage").sum()
        records.append(row)

    counts = pd.DataFrame(records).set_index("era")

    print("  Milestone counts by era  (seasons Italy reached each round):")
    cols = ["n_seasons"] + milestones + ["stage_only"]
    header = f"  {'Era':<22}  {'N':>9}  " + \
             "  ".join(f"{m:>14}" for m in milestones) + \
             f"  {'Stage only':>12}"
    print(header)
    print("  " + "-" * (len(header) - 2))
    for era in ERA_ORDER:
        if era not in counts.index:
            continue
        r   = counts.loc[era]
        row = f"  {era:<22}  {int(r['n_seasons']):>9}  "
        row += "  ".join(f"{int(r[m]):>14}" for m in milestones)
        row += f"  {int(r['stage_only']):>12}"
        print(row)
    print()
    return counts

In [17]:
depth = depth_summary_table(italy_csh, round_col="highest round")


  Tournament depth by era  (Italy)
  Era                      N   Avg ord  Avg round         Best round      
  ------------------------------------------------------------------------
  Pre-Golden Era          18      3.11  Semi Finals       Final           
  Golden Era              13      3.77  Final             Final           
  Post-Golden Era         26      2.88  Semi Finals       Final           



What this table is telling us. What we have done is to assign for each stage a number in this way: 

- "preliminarly stage" : 0
- "Round of 16" : 1 
- "Quarter Finals" : 2 
- "Semi Finals" : 3  
- "Final" : 4

And then considered what was, for that season, the deepest stage the Italian club reached. We then calculated the mean (`Avg ord`) and the rounded to find the `Avg round` (that is a category). The `Avg ord` is the (float) mean, the `Avg round` is the category it belongs to. This mean that in Pre and Post Golden era at least on Italian Club team reached, on average, the Semi Finals in the tournment, while during the Golden Era they reached the Final (in a quite strong way, with an mean of 3.77)


In [18]:
milest = milestone_counts(italy_csh, round_col="highest round")

  Milestone counts by era  (seasons Italy reached each round):
  Era                             N           Final     Semi Finals  Quarter Finals     Round of 16    Stage only
  ---------------------------------------------------------------------------------------------------------------
  Pre-Golden Era                 18               8               6               3               0             1
  Golden Era                     13              11               1               1               0             0
  Post-Golden Era                26              11               6               6               1             2



This table counts the number of different stages reached in each year belonging to the different eras. During the 13 years of the Golden Era, only one year all Italian Clubs reached only the Quarter Finals.

In [19]:
italy_csh[italy_csh['era'] == 'Golden Era']

,season,country,highest round,competition,era
18,1987/88,Ita,Semi Finals,[CW],Golden Era
19,1988/89,Ita,Final,"[CL, CW, EL]",Golden Era
20,1989/90,Ita,Final,"[CL, CW, EL]",Golden Era
21,1990/91,Ita,Final,[EL],Golden Era
22,1991/92,Ita,Final,"[CL, EL]",Golden Era
23,1992/93,Ita,Final,"[CL, CW, EL]",Golden Era
24,1993/94,Ita,Final,"[CL, CW, EL]",Golden Era
25,1994/95,Ita,Final,"[CL, EL]",Golden Era
26,1995/96,Ita,Final,[CL],Golden Era
27,1996/97,Ita,Final,"[CL, EL]",Golden Era


Well, not surprisignly, we framed the golden era between the only Semi Finals and Final...

### e. Statistical Test

Finally we reached where the fun begins - yes. It's Friday. Napoli just won against Cagliari and I am really having fun by cleaning all the mess I produced in the latest days trying to make sense in all the data and results produced.

We produces measurements in the previous steps and now it's time to ask ourselves if the difference between eras are really statistically significant and, possibly, if strong enough to make clear conclusions.

For each metric, we will test whether the Golden Era has values significantly greater than each of the other two eras.

Tests used:
- Mann-Whitney $U$  (non-parametric, no normality assumption)
    - $H_{0}$ : distributions are equal
    - $H_{A}$ : Golden Era > other era   (one-tailed, alternative='greater')
    - $\alpha = 0.05$  (flagged at 0.10 as a weak signal)
- Cohen's $d$  (effect size, pooled SD)
    - $|d| < 0.2$ means negligible
    - $|d| < 0.5$|d| means small
    - $|d| < 0.8$  means  medium
    - $|d| \geq 0.8$  means  large

Minimum sample size: 3 per era (pairs with fewer seasons are skipped with a printed notice rather than producing spurious p-values)


In [20]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu


# Helpers

def _cohen_d(a: np.ndarray, b: np.ndarray) -> float:
    """Pooled-SD Cohen's d (positive = a > b)."""
    var_a = np.var(a, ddof=1) if len(a) > 1 else 0.0
    var_b = np.var(b, ddof=1) if len(b) > 1 else 0.0
    pooled = np.sqrt((var_a + var_b) / 2)
    return (np.mean(a) - np.mean(b)) / pooled if pooled > 0 else 0.0


def _sig_stars(p: float) -> str:
    if p < 0.01:  return "***"
    if p < 0.05:  return "**"
    if p < 0.10:  return "*"
    return "ns"


def _effect_label(d: float) -> str:
    ad = abs(d)
    if ad < 0.2:  return "negligible"
    if ad < 0.5:  return "small"
    if ad < 0.8:  return "medium"
    return "large"


MIN_N = 3   # minimum seasons per era to run a test


# ── Main test functions ───────────────────────────────────────────────────────

def test_match_metrics(italy_cs: pd.DataFrame) -> pd.DataFrame:
    """
    Run Mann-Whitney U + Cohen's d for win_rate, ppg3_pt, gdpg_pt:
      Golden Era  vs  Pre-Golden Era
      Golden Era  vs  Post-Golden Era

    Parameters
    ----------
    italy_cs : Italy country_stats with 'era' and derived metric columns
               (output of step1b add_derived_metrics)

    Returns
    -------
    results DataFrame with one row per (metric × comparison)
    """
    
    METRICS = {
    "win_rate" : "Win rate",
    "ppg3_pt"  : "PPG (3pt) per team",
    "gdpg_pt"  : "GDpG per team",
    }
    
    golden = italy_cs[italy_cs["era"] == "Golden Era"]
    rows   = []

    print("\n  ── Match-level metric tests ──────────────────────────────────")
    print(f"  {'Metric':<22}  {'vs':<22}  "
          f"{'U':>7}  {'p':>7}  {'sig':>4}  "
          f"{'d':>6}  {'effect':<12}")
    print("  " + "-" * 80)

    for col, label in METRICS.items():
        g_vals = golden[col].dropna().values
        for other_era in ["Pre-Golden Era", "Post-Golden Era"]:
            o_vals = italy_cs[italy_cs["era"] == other_era][col].dropna().values
            row    = {"metric": label, "comparison": f"vs {other_era}",
                      "n_golden": len(g_vals), "n_other": len(o_vals)}

            if len(g_vals) < MIN_N or len(o_vals) < MIN_N:
                row.update(U=np.nan, p=np.nan, sig="skip", d=np.nan,
                           effect="too few seasons")
                print(f"  {label:<22}  {other_era:<22}  "
                      f"  SKIP (n_golden={len(g_vals)}, n_other={len(o_vals)})")
                rows.append(row)
                continue

            stat, p = mannwhitneyu(g_vals, o_vals, alternative="greater")
            d       = _cohen_d(g_vals, o_vals)
            sig     = _sig_stars(p)
            eff     = _effect_label(d)

            row.update(U=stat, p=p, sig=sig, d=d, effect=eff)
            rows.append(row)
            print(f"  {label:<22}  {other_era:<22}  "
                  f"{stat:>7.0f}  {p:>7.4f}  {sig:>4}  "
                  f"{d:>+6.2f}  {eff:<12}")

    print()
    return pd.DataFrame(rows)


def test_depth(italy_csh: pd.DataFrame,
               round_col: str = "highest_round") -> pd.DataFrame:
    """
    Run Mann-Whitney U + Cohen's d on the ordinal round depth:
      Golden Era  vs  Pre-Golden Era
      Golden Era  vs  Post-Golden Era

    Parameters
    ----------
    italy_csh : Italy country_season_highest with 'era' and 'highest_round'

    Returns
    -------
    results DataFrame with one row per comparison
    """
    # Apply numeric encoding (same as step1d)
    ROUND_GROUPS = ["stage", "Round of 16", "Quarter Finals", "Semi Finals", "Final"]
    
    df = italy_csh.copy()
    df[round_col] = pd.Categorical(df[round_col],
                                   categories=ROUND_GROUPS, ordered=True)
    df["round_ord"] = df[round_col].cat.codes.replace(-1, pd.NA)

    golden = df[df["era"] == "Golden Era"]
    rows   = []

    print("  ── Tournament depth tests ────────────────────────────────────")
    print(f"  {'Metric':<22}  {'vs':<22}  "
          f"{'U':>7}  {'p':>7}  {'sig':>4}  "
          f"{'d':>6}  {'effect':<12}")
    print("  " + "-" * 80)

    g_vals = golden["round_ord"].dropna().values
    for other_era in ["Pre-Golden Era", "Post-Golden Era"]:
        o_vals = df[df["era"] == other_era]["round_ord"].dropna().values
        row    = {"metric": "Round depth (ord)", "comparison": f"vs {other_era}",
                  "n_golden": len(g_vals), "n_other": len(o_vals)}

        if len(g_vals) < MIN_N or len(o_vals) < MIN_N:
            row.update(U=np.nan, p=np.nan, sig="skip", d=np.nan,
                       effect="too few seasons")
            print(f"  {'Round depth':<22}  {other_era:<22}  "
                  f"  SKIP (n={len(g_vals)} / {len(o_vals)})")
            rows.append(row)
            continue

        stat, p = mannwhitneyu(g_vals, o_vals, alternative="greater")
        d       = _cohen_d(g_vals, o_vals)
        sig     = _sig_stars(p)
        eff     = _effect_label(d)

        row.update(U=stat, p=p, sig=sig, d=d, effect=eff)
        rows.append(row)
        print(f"  {'Round depth (ord)':<22}  {other_era:<22}  "
              f"{stat:>7.0f}  {p:>7.4f}  {sig:>4}  "
              f"{d:>+6.2f}  {eff:<12}")

    print()
    return pd.DataFrame(rows)


def full_stat_report(match_results: pd.DataFrame,
                     depth_results: pd.DataFrame) -> pd.DataFrame:
    """
    Combine match-level and depth test results into one summary table
    and print a plain-language interpretation.
    """
    combined = pd.concat([match_results, depth_results], ignore_index=True)

    print("  ── Interpretation guide ──────────────────────────────────────")
    print("  sig: *** p<0.01  ** p<0.05  * p<0.10  ns = not significant")
    print("  d  : + = Golden Era higher | effect: small/medium/large\n")

    sig_count = (combined["sig"].isin(["***","**","*"])).sum()
    total     = combined["sig"].notna().sum()
    print(f"  {sig_count} of {total} tests show a significant result "
          f"(α ≤ 0.10) in favour of the Golden Era.\n")

    return combined

In [21]:
match_res  = test_match_metrics(italy_cs)


  ── Match-level metric tests ──────────────────────────────────
  Metric                  vs                            U        p   sig       d  effect      
  --------------------------------------------------------------------------------
  Win rate                Pre-Golden Era              176   0.0091   ***   +1.09  large       
  Win rate                Post-Golden Era             300   0.0001   ***   +1.47  large       
  PPG (3pt) per team      Pre-Golden Era               98   0.7766    ns   -0.45  small       
  PPG (3pt) per team      Post-Golden Era             264   0.0024   ***   +1.08  large       
  GDpG per team           Pre-Golden Era              154   0.0720     *   +0.56  medium      
  GDpG per team           Post-Golden Era             307   0.0000   ***   +1.57  large       



Our gut feeling about the higher performances during the Golden era, if we consider the ***match performances***, is confirmed statistically. Form the above Table:

- The Win Rate is definetly higher in the Golden Era with respect to the other eras, with higher $d$ effect size meaning that the evidence is strong.
- The PPG (3pt) is the weaker indicator but it's not surprising: during the pre-golden era, few teams would partecipate (and winning was counted 2 points)
- The Goal Difference per Team it's again statistically different in favour of the golden era, especially in the Post era (note: Post-Golden era the number of teams was different but not that much)

In [22]:
depth_res  = test_depth(italy_csh, round_col="highest round")

  ── Tournament depth tests ────────────────────────────────────
  Metric                  vs                            U        p   sig       d  effect      
  --------------------------------------------------------------------------------
  Round depth (ord)       Pre-Golden Era              164   0.0171    **   +0.75  medium      
  Round depth (ord)       Post-Golden Era             244   0.0071   ***   +0.91  large       



And also the Tournment depth test confirm a superiority during the Golden Era.